In [1]:
import importlib
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "analysis" / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f"Could not locate project root from {Path.cwd().resolve()}")
DROPBOX_ROOT = Path("/Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC")
sys.path.append(str(PROJECT_ROOT / "analysis" / "src"))
sys.path.append(str(PROJECT_ROOT / "analysis" / "gibbs"))

DATA_DIR = PROJECT_ROOT / "data"
IDATA_ROOT = DROPBOX_ROOT / "results" / "idata"
TEX_ROOT_DIR = PROJECT_ROOT / "results" / "tex" / "table"
FIG_ROOT = PROJECT_ROOT / "results" / "fig"

# Change DROPBOX_ROOT above if your Dropbox location differs.
# IDATA files will be saved to and loaded from IDATA_ROOT.
from func_data_build import build_dataset
from func_gibbs.gibbs_notebook_utils import display_hmc_results, display_hmc_posterior_prior, save_idata_map
from func_gibbs.gibbs_wrappers import draws_to_idata
import func_gibbs.gibbs_wrappers as gibbs_module
from func_gibbs.gibbs_wrappers import run_hsa_dynamic

data_dir = DATA_DIR
idat_dir = IDATA_ROOT
tex_root_dir = TEX_ROOT_DIR
base_fig_dir = FIG_ROOT

## 0. load data
data = build_dataset(data_dir)
data["DATE"] = pd.to_datetime(data.index)

## make model-specific maximum available sample
def make_model_sample(data, spec, inflation_specs, include_N=True):
    infl = inflation_specs[spec["inflation"]]
    gap = spec["gap"]

    required_cols = [
        infl["pi"],
        infl["pi_prev"],
        infl["pi_expect"],
        gap,
        f"{gap}_prev",
    ]

    if include_N:
        required_cols.append(spec["N"])

    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing columns for {spec['model_id']}: {missing_cols}")

    sample = (
        data[required_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    if sample.empty:
        raise ValueError(f"No valid sample for {spec['model_id']}")

    return sample


In [2]:
## model specifications
inflation_specs = {
    "ppi": {
        "pi": "pi_ppi",
        "pi_prev": "pi_ppi_prev",
        "pi_expect": "Epi_spf_gdp",
    },
    "cpi": {
        "pi": "pi_cpi",
        "pi_prev": "pi_cpi_prev",
        "pi_expect": "Epi_spf_cpi",
    },
}

model_grid = [
    {
        "model_id": "S1_U_G",
        "set": 1,
        "inflation": "ppi",
        "gap": "unemp_gap",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S1_U_T",
        "set": 1,
        "inflation": "ppi",
        "gap": "unemp_gap",
        "N": "N_TNIC",
    },
    {
        "model_id": "S1_Y_G",
        "set": 1,
        "inflation": "ppi",
        "gap": "output_gap_BN",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S1_Y_T",
        "set": 1,
        "inflation": "ppi",
        "gap": "output_gap_BN",
        "N": "N_TNIC",
    },
    {
        "model_id": "S2_U_G",
        "set": 2,
        "inflation": "cpi",
        "gap": "unemp_gap",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S2_U_T",
        "set": 2,
        "inflation": "cpi",
        "gap": "unemp_gap",
        "N": "N_TNIC",
    },
    {
        "model_id": "S2_Y_G",
        "set": 2,
        "inflation": "cpi",
        "gap": "output_gap_BN",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S2_Y_T",
        "set": 2,
        "inflation": "cpi",
        "gap": "output_gap_BN",
        "N": "N_TNIC",
    },
]

## 4. check sample periods

sample_info = []

for spec in model_grid:
    sample = make_model_sample(
        data=data,
        spec=spec,
        inflation_specs=inflation_specs,
        include_N=True,
    )

    sample_info.append({
        "model_id": spec["model_id"],
        "set": spec["set"],
        "inflation": spec["inflation"],
        "gap": spec["gap"],
        "N": spec["N"],
        "start": sample.index.min(),
        "end": sample.index.max(),
        "T": len(sample),
    })

sample_info_df = pd.DataFrame(sample_info)
display(sample_info_df)


,model_id,set,inflation,gap,N,start,end,T
0,S1_U_G,1,ppi,unemp_gap,N_Gustavo,1974-03-31 23:59:59.999999999,2012-12-31 23:59:59.999999999,155
1,S1_U_T,1,ppi,unemp_gap,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
2,S1_Y_G,1,ppi,output_gap_BN,N_Gustavo,1974-03-31 23:59:59.999999999,2012-12-31 23:59:59.999999999,155
3,S1_Y_T,1,ppi,output_gap_BN,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
4,S2_U_G,2,cpi,unemp_gap,N_Gustavo,1981-09-30 23:59:59.999999999,2012-12-31 23:59:59.999999999,126
5,S2_U_T,2,cpi,unemp_gap,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
6,S2_Y_G,2,cpi,output_gap_BN,N_Gustavo,1981-09-30 23:59:59.999999999,2012-12-31 23:59:59.999999999,126
7,S2_Y_T,2,cpi,output_gap_BN,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140


In [3]:

## 5. prior distributions and MCMC parameters

seed = 0
n_iter = 14000
burn = 2000
thin = 5

prior_specs = {
    "alpha": (0.5, 1),
    "kappa": (0.1, 1),
    "theta": (0.1, 1),
    "phi_1": (0.7, 1),
    "rho_1": (0.2, 1),
    "rho_2": (0.2, 1),
    "rho": (0.0, 1),
}

display(pd.DataFrame([
    {"parameter": key, "prior_mean": value[0], "prior_sd": value[1]}
    for key, value in prior_specs.items()
]))


## 6. Gibbs MCMC: HSA dynamic models

gibbs_module = importlib.reload(gibbs_module)
run_hsa_dynamic = gibbs_module.run_hsa_dynamic
draws_to_idata = gibbs_module.draws_to_idata

from types import SimpleNamespace
import xarray as xr

dict_draws = {}
dict_idata = {}

for spec in model_grid:
    sample = make_model_sample(
        data=data,
        spec=spec,
        inflation_specs=inflation_specs,
        include_N=True,
    )

    infl = inflation_specs[spec["inflation"]]
    gap = spec["gap"]
    N_col = spec["N"]

    pi = np.asarray(sample[infl["pi"]], dtype=float)
    pi_prev = np.asarray(sample[infl["pi_prev"]], dtype=float)
    pi_expect = np.asarray(sample[infl["pi_expect"]], dtype=float)

    x = np.asarray(sample[gap], dtype=float)
    x_prev = np.asarray(sample[f"{gap}_prev"], dtype=float)
    N = np.asarray(sample[N_col], dtype=float)

    for orth in (False, True):
        label = "corr" if not orth else "uncorr"
        model_name = f"{spec['model_id']}_HSA_dynamic_{label}"
        out_path = idat_dir / f"{model_name}.nc"

        if out_path.exists():
            print("Loading", model_name, "| found:", out_path)
            posterior = xr.open_dataset(out_path, group="posterior").load()
            dict_idata[model_name] = SimpleNamespace(posterior=posterior)
            continue

        print(
            "Running",
            model_name,
            "| sample:",
            sample.index.min(),
            "to",
            sample.index.max(),
            "| T =",
            len(sample),
        )

        draws = run_hsa_dynamic(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            N=N,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=orth,
        )

        idata = draws_to_idata(draws)
        dict_draws[model_name] = draws
        dict_idata[model_name] = idata
        save_idata_map({model_name: idata}, idat_dir)
        print("Saved", model_name, "to", out_path)

list(dict_idata.keys())

print(f"Prepared {len(dict_idata)} idata objects from {idat_dir}")


,parameter,prior_mean,prior_sd
0,alpha,0.5,1
1,kappa,0.1,1
2,theta,0.1,1
3,phi_1,0.7,1
4,rho_1,0.2,1
5,rho_2,0.2,1
6,rho,0.0,1


Running S1_U_G_HSA_dynamic_corr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Saved S1_U_G_HSA_dynamic_corr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_G_HSA_dynamic_corr.nc
Running S1_U_G_HSA_dynamic_uncorr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Saved S1_U_G_HSA_dynamic_uncorr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_G_HSA_dynamic_uncorr.nc
Running S1_U_T_HSA_dynamic_corr | sample: 1988-03-31 23:59:59.999999999 to 2022-12-31 23:59:59.999999999 | T = 140
Saved S1_U_T_HSA_dynamic_corr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_T_HSA_dynamic_corr.nc
Running S1_U_T_HSA_dynamic_uncorr | sample: 1988-03-31 23:59:59.999999999 to 2022-12-31 23:59:59.999999999 | T = 140
Saved S1_U_T_HSA_dynamic_uncorr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_T_HSA_dynamic_uncorr.nc


KeyboardInterrupt: 

In [ ]:
## posterior summary
display_hmc_results(
    dict_idata,
    prior_specs,
    models_to_show=list(dict_idata.keys()),
    params=("alpha", "kappa", "theta", "phi_1", "rho_1", "rho_2", "rho", "sigma_e", "sigma_zeta", "sigma_u", "sigma_eps"),
    title="Posterior summary for HSA dynamic models",
)


,model,alpha,kappa,theta,phi_1,rho_1,rho_2,rho,sigma_e,sigma_zeta,sigma_u,sigma_eps
0,S1_U_G_HSA_dynamic_corr,0.7832,0.2427,0.0553,0.9388,1.2247,-0.2423,0.1049,2.4948,0.6521,1.1964,1.1526
1,S1_U_G_HSA_dynamic_uncorr,0.7789,0.3428,0.0408,0.9384,1.2171,-0.2256,0.0000,2.4873,0.6519,1.2219,1.1461
2,S1_U_T_HSA_dynamic_corr,0.8493,-0.2152,0.0158,0.8526,1.1495,-0.1686,0.2670,3.0176,0.9752,1.1975,1.1526
3,S1_U_T_HSA_dynamic_uncorr,0.8503,0.0095,0.0015,0.8529,1.1807,-0.2058,0.0000,2.9795,0.9748,1.2667,1.2011
4,S1_Y_G_HSA_dynamic_corr,0.8248,0.3232,0.0420,0.8982,1.3032,-0.3169,0.0244,2.4826,0.7081,1.1526,1.0592
5,S1_Y_G_HSA_dynamic_uncorr,0.8392,0.2928,0.0108,0.9007,1.1720,-0.1902,0.0000,2.4841,0.7062,1.1766,1.1772
6,S1_Y_T_HSA_dynamic_corr,0.8371,0.3280,0.0389,0.7302,1.2105,-0.2414,0.2050,2.8976,0.9761,1.2223,1.1226
7,S1_Y_T_HSA_dynamic_uncorr,0.7934,0.7216,0.1103,0.7342,1.1830,-0.2111,0.0000,2.8021,0.9785,1.2247,1.1844
8,S2_U_G_HSA_dynamic_corr,0.6456,0.0988,0.0263,0.9569,1.1682,-0.2027,0.0531,0.6809,0.5942,1.2582,1.2771
9,S2_U_G_HSA_dynamic_uncorr,0.6380,0.1340,0.0118,0.9576,1.2938,-0.2989,0.0000,0.6750,0.5938,1.2464,1.1439


Posterior summary for HSA dynamic models
                         model       param      mean    ci_2.5   ci_97.5
0      S1_U_G_HSA_dynamic_corr       alpha  0.783155  0.683607  0.883405
1      S1_U_G_HSA_dynamic_corr       kappa  0.242711 -0.060107  0.571121
2      S1_U_G_HSA_dynamic_corr       theta  0.055264 -0.028084  0.138238
3      S1_U_G_HSA_dynamic_corr       phi_1  0.938751  0.882485  0.994653
4      S1_U_G_HSA_dynamic_corr       rho_1  1.224676  0.914185  1.495762
..                         ...         ...       ...       ...       ...
171  S2_Y_T_HSA_dynamic_uncorr         rho  0.000000  0.000000  0.000000
172  S2_Y_T_HSA_dynamic_uncorr     sigma_e  0.670170  0.590244  0.759858
173  S2_Y_T_HSA_dynamic_uncorr  sigma_zeta  0.978398  0.872604  1.104983
174  S2_Y_T_HSA_dynamic_uncorr     sigma_u  1.273943  0.945868  1.689300
175  S2_Y_T_HSA_dynamic_uncorr   sigma_eps  1.161361  0.883550  1.512781

[176 rows x 5 columns]
